# Wikimedia Bronze Layer

Generates raw Wikimedia event data via `produce()` (defined in `Producer_streaming_engine.ipynb`), converts it into a Spark DataFrame, and persists it to the bronze Delta table `workspace.bronze.wikimedia_bronze`. The `read_raw()` helper adds an `_ingest_time` timestamp, casts the `timestamp` column, caches the DataFrame, and appends to `{catalog}.{bronze_schema}.{bronze_table}` using a batch write. Configurations — catalog, schema, and table names — are sourced from `params_config` via `%run`.

### The universal pattern for an event stream with varying optional fields is:

- Declare one explicit, nullable, superset schema covering every field that can ever appear across all event types. This solves “some fields don’t show up in some files” — by design, not by accident.

- Keep cloudFiles.schemaEvolutionMode = "rescue" as a safety net, not your primary mechanism — it should catch truly unexpected stuff (a brand-new field Wikimedia adds tomorrow), not routinely absorb fields you already know about.\
`(pure schema inference, no explicit schema) is the wrong default for this kind of source — inference is unstable (can flip types batch-to-batch, which is literally what caused your original timestamp failure) and gives you no durable contract to build Silver/Gold on top of.`

In [0]:
%run ./params_config

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

schema = StructType([
    StructField("$schema", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("id", LongType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True),
    ]), True),
    StructField("log_action", StringType(), True),
    StructField("log_action_comment", StringType(), True),
    StructField("log_id", LongType(), True),
    StructField("log_params", StringType(), True),
    StructField("log_type", StringType(), True),
    StructField("meta", StructType([
        StructField("domain", StringType(), True),
        StructField("dt", StringType(), True),
        StructField("id", StringType(), True),
        StructField("offset", LongType(), True),
        StructField("partition", LongType(), True),
        StructField("request_id", StringType(), True),
        StructField("stream", StringType(), True),
        StructField("topic", StringType(), True),
        StructField("uri", StringType(), True),
    ]), True),
    StructField("minor", BooleanType(), True),
    StructField("namespace", LongType(), True),
    StructField("notify_url", StringType(), True),
    StructField("parsedcomment", StringType(), True),
    StructField("patrolled", BooleanType(), True),
    StructField("revision", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True),
    ]), True),
    StructField("server_name", StringType(), True),
    StructField("server_script_path", StringType(), True),
    StructField("server_url", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("title", StringType(), True),
    StructField("title_url", StringType(), True),
    StructField("type", StringType(), True),
    StructField("user", StringType(), True),
    StructField("wiki", StringType(), True),
])

In [0]:
import logging
from pyspark.sql.functions import current_timestamp

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")


def auto_loader(schema, SOURCE_PATH, schema_location_bronze, bronze_schema, bronze_table, catalog, checkpoint_bronze):
    # Auto Loader ingest -> add ingestion timestamp -> stream to Delta bronze table
    df = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(schema)
            .option("cloudFiles.schemaLocation", schema_location_bronze)
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .option("rescuedDataColumn", "_rescued_data")
            .option("badRecordsPath", f"{SOURCE_PATH.rstrip('/')}_corrupt_records")
            .load(SOURCE_PATH))

    enriched_df = df.withColumn("ingested_at", current_timestamp())

    target_table = f"{catalog}.{bronze_schema}.{bronze_table}"
    print(f"[auto_loader] starting stream -> {target_table}")
    logging.info(f"auto_loader starting: source={SOURCE_PATH} target={target_table}")

    query = (enriched_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_bronze)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{bronze_schema}.{bronze_table}"))

    query.awaitTermination()

    row_count = spark.table(target_table).count()
    print(f"[auto_loader] stream finished, table row count: {row_count}")
    logging.info(f"auto_loader complete: {target_table} row_count={row_count}")

    return query


In [0]:

auto_loader(schema, SOURCE_PATH, schema_location_bronze, bronze_schema, bronze_table, catalog, checkpoint_bronze)
spark.streams.awaitAnyTermination()